# Databricks SQL Warehouse Cost Optimization — End-to-EndCollects every required signal from system tables, derives the decision metrics,and emits a priced recommendation per warehouse.**Architecture: fully deterministic.** No LLM is on the decision path. Cost figuresare arithmetic, thresholds are explicit constants, and the same input alwaysproduces the same output. An optional rationale is generated from a template.**Run on:** any Databricks cluster / SQL warehouse with `USE CATALOG system` access.---### Pipeline| Stage | What it does ||---|---|| 0 | Params + preflight (which system tables exist) || 1 | Extract: warehouse config, billing, lifecycle events, query history || 2 | Derive: cluster-hours, duty cycle, concurrency, spill/queue rates || 3 | Decide: enumerate candidate configs, score, pick cheapest feasible || 4 | Guardrails: suppress recommendations that a missing signal would invert || 5 | Output: rationale + Delta table |### Known gaps (see `docs/sql-warehouse-signals.md`)- **EC2/EBS cost for Classic warehouses is not in system tables.** Supply it via  `cur_cost_table` or the `classic_infra_multiplier` fallback, otherwise every  Classic-vs-Serverless comparison is biased toward Classic.- Node CPU/memory is not exposed for SQL warehouses — sizing is driven by  `spilled_local_bytes`, not utilization.

## 0 · Parameters

In [ ]:
dbutils.widgets.text("lookback_days", "30", "Lookback window (days)")dbutils.widgets.text("output_table", "", "Output Delta table (blank = display only)")dbutils.widgets.text("cur_cost_table", "", "AWS CUR cost table (warehouse_id, usage_date, infra_cost_usd)")dbutils.widgets.text("classic_infra_multiplier", "1.0", "Fallback: Classic infra cost as x DBU cost")dbutils.widgets.text("workspace_filter", "", "Restrict to workspace_id (blank = all)")LOOKBACK_DAYS   = int(dbutils.widgets.get("lookback_days"))OUTPUT_TABLE    = dbutils.widgets.get("output_table").strip()CUR_TABLE       = dbutils.widgets.get("cur_cost_table").strip()INFRA_MULT      = float(dbutils.widgets.get("classic_infra_multiplier"))WORKSPACE       = dbutils.widgets.get("workspace_filter").strip()print(f"lookback={LOOKBACK_DAYS}d  cur_table={CUR_TABLE or 'NONE'}  infra_mult={INFRA_MULT}")

In [ ]:
from pyspark.sql import functions as Ffrom datetime import datetime, timedeltaimport pandas as pd, math, jsonspark.conf.set("spark.sql.ansi.enabled", "false")END_TS   = datetime.utcnow()START_TS = END_TS - timedelta(days=LOOKBACK_DAYS)START, END = START_TS.strftime("%Y-%m-%d %H:%M:%S"), END_TS.strftime("%Y-%m-%d %H:%M:%S")HOURS_IN_WINDOW = LOOKBACK_DAYS * 24WS_PRED = f"AND workspace_id = '{WORKSPACE}'" if WORKSPACE else ""# ---- Structural constants -------------------------------------------------# DBU/hr per cluster by T-shirt size. Verify against system.billing.list_prices.SIZE_DBU = {"2X-Small": 4, "X-Small": 6, "Small": 12, "Medium": 24, "Large": 40,            "X-Large": 80, "2X-Large": 144, "3X-Large": 272, "4X-Large": 528}SIZE_ORDER = list(SIZE_DBU.keys())# Databricks admits ~10 concurrent queries per cluster before queuing.# Not user-tunable (unlike Snowflake MAX_CONCURRENCY_LEVEL). Verify for Serverless.QUERIES_PER_CLUSTER = 10# ---- Decision thresholds (tune here; every rule reads from this dict) -----TH = {    "duty_cycle_serverless_max": 0.30,   # below -> Serverless favoured    "duty_cycle_classic_min":    0.70,   # above -> Classic may win    "spill_rate_upsize":         0.10,   # >10% of queries spilling -> size up    "spill_rate_safe_downsize":  0.01,   # must be under this to downsize    "queue_rate_scale_out":      0.05,   # >5% queued -> add clusters    "idle_ratio_autostop":       0.40,   # >40% idle -> tighten auto_stop    "cluster_headroom_shrink":   0.50,   # p95 <= 50% of max -> shrink max    "time_at_max_raise":         0.10,   # >10% pinned at max -> raise max    "min_queries_for_high_conf": 100,    "min_days_for_high_conf":    21,    "cold_start_min_classic":    5.0,    # min startup billed on Classic (minutes)    "cold_start_min_serverless": 0.2,}print(f"window: {START} -> {END}")

## 0b · Preflight — which system tables are actually available

In [ ]:
REQUIRED = {    "warehouses":       "system.compute.warehouses",    "warehouse_events": "system.compute.warehouse_events",    "usage":            "system.billing.usage",    "list_prices":      "system.billing.list_prices",    "query_history":    "system.query.history",}AVAIL = {}for key, tbl in REQUIRED.items():    try:        spark.sql(f"SELECT 1 FROM {tbl} LIMIT 1").collect()        AVAIL[key] = True    except Exception as e:        AVAIL[key] = False        print(f"  UNAVAILABLE  {tbl}\n     -> {str(e)[:160]}")for k, v in AVAIL.items():    print(f"  {'OK        ' if v else 'MISSING   '} {REQUIRED[k]}")blocking = [k for k in ("usage", "list_prices", "query_history") if not AVAIL[k]]if blocking:    raise RuntimeError(f"Cannot proceed without: {blocking}")if not AVAIL["warehouses"]:    print("\nWARN: system.compute.warehouses missing (newest table, may be preview/region-gated).")    print("      Config signals (size/min/max/auto_stop) must come from the SQL Warehouses API.")if not AVAIL["warehouse_events"]:    print("\nWARN: warehouse_events missing -> no cluster_hours, no duty cycle.")    print("      Cost will be attributed from billing only; cluster tuning disabled.")

In [ ]:
def cols_of(table):    """Return the actual column set - query.history names have shifted across releases."""    try:        return {f.name for f in spark.table(table).schema.fields}    except Exception:        return set()def pick(available, *candidates, default="NULL"):    """First candidate that actually exists, else a NULL literal placeholder."""    for c in candidates:        if c in available:            return c    return defaultQH = cols_of("system.query.history")C_READ_ROWS  = pick(QH, "read_rows", "rows_read_count")C_PROD_ROWS  = pick(QH, "produced_rows", "rows_produced_count")C_PRUNED     = pick(QH, "pruned_files", "pruned_files_count")C_READ_FILES = pick(QH, "read_files", "read_files_count")C_SPILL      = pick(QH, "spilled_local_bytes")HAS_SPILL = C_SPILL != "NULL"if not HAS_SPILL:    print("CRITICAL: spilled_local_bytes unavailable -> ALL downsize recommendations will be suppressed.")print(f"resolved: read_rows={C_READ_ROWS} produced_rows={C_PROD_ROWS} spill={C_SPILL}")

## 1 · Extract### 1a · Warehouse inventory (Tier 0 — config)`system.compute.warehouses` is SCD2; take the latest non-deleted row per warehouse.

In [ ]:
if AVAIL["warehouses"]:    wh = spark.sql(f"""        WITH ranked AS (          SELECT *, ROW_NUMBER() OVER (PARTITION BY warehouse_id ORDER BY change_time DESC) rn          FROM system.compute.warehouses          WHERE delete_time IS NULL {WS_PRED}        )        SELECT warehouse_id, workspace_id, warehouse_name,               warehouse_type, warehouse_size, warehouse_channel,               CAST(min_clusters AS INT) min_clusters,               CAST(max_clusters AS INT) max_clusters,               CAST(auto_stop_minutes AS INT) auto_stop_minutes,               change_time AS config_as_of        FROM ranked WHERE rn = 1    """)else:    # Fallback: identify warehouses from billing only. Config fields unknown.    wh = spark.sql(f"""        SELECT DISTINCT usage_metadata.warehouse_id AS warehouse_id, workspace_id,               CAST(NULL AS STRING) warehouse_name,               CASE WHEN sku_name ILIKE '%SERVERLESS%' THEN 'SERVERLESS' ELSE 'CLASSIC' END warehouse_type,               CAST(NULL AS STRING) warehouse_size, CAST(NULL AS STRING) warehouse_channel,               CAST(NULL AS INT) min_clusters, CAST(NULL AS INT) max_clusters,               CAST(NULL AS INT) auto_stop_minutes, CAST(NULL AS TIMESTAMP) config_as_of        FROM system.billing.usage        WHERE usage_metadata.warehouse_id IS NOT NULL          AND usage_start_time >= '{START}' {WS_PRED}    """)wh.createOrReplaceTempView("wh_config")print(f"warehouses: {wh.count()}")display(wh.limit(20))

### 1b · Config driftConsecutive SCD2 rows reveal the "temporary" resize nobody reverted.

In [ ]:
if AVAIL["warehouses"]:    drift = spark.sql(f"""        WITH h AS (          SELECT warehouse_id, change_time, warehouse_size, min_clusters, max_clusters, auto_stop_minutes,                 LAG(warehouse_size)     OVER (PARTITION BY warehouse_id ORDER BY change_time) p_size,                 LAG(max_clusters)       OVER (PARTITION BY warehouse_id ORDER BY change_time) p_max,                 LAG(auto_stop_minutes)  OVER (PARTITION BY warehouse_id ORDER BY change_time) p_stop          FROM system.compute.warehouses          WHERE change_time >= '{START}' {WS_PRED}        )        SELECT warehouse_id, change_time,               CONCAT_WS(', ',                 CASE WHEN warehouse_size    <> p_size THEN CONCAT('size ',      p_size, '->', warehouse_size) END,                 CASE WHEN max_clusters      <> p_max  THEN CONCAT('max_cl ',    p_max,  '->', max_clusters) END,                 CASE WHEN auto_stop_minutes <> p_stop THEN CONCAT('auto_stop ', p_stop, '->', auto_stop_minutes) END               ) AS change        FROM h        WHERE p_size IS NOT NULL          AND (warehouse_size <> p_size OR max_clusters <> p_max OR auto_stop_minutes <> p_stop)        ORDER BY change_time DESC    """)    drift.createOrReplaceTempView("wh_drift")    print(f"config changes in window: {drift.count()}")    display(drift.limit(20))else:    spark.sql("SELECT CAST(NULL AS STRING) warehouse_id, CAST(NULL AS STRING) change LIMIT 0") \         .createOrReplaceTempView("wh_drift")

### 1c · Cost (Tier 1)DBUs from `billing.usage`, priced from `list_prices` joined on the effective date range.> `list_prices` is **list** price. If you have a negotiated rate, override it here —> otherwise every savings figure is an over-estimate.

In [ ]:
cost = spark.sql(f"""    SELECT u.usage_metadata.warehouse_id AS warehouse_id,           MAX(u.sku_name)                                   AS sku_name,           SUM(u.usage_quantity)                             AS dbus,           SUM(u.usage_quantity * CAST(p.pricing.default AS DOUBLE)) AS dbu_cost_usd,           COUNT(DISTINCT u.usage_date)                      AS billed_days    FROM system.billing.usage u    JOIN system.billing.list_prices p      ON u.sku_name = p.sku_name     AND u.usage_end_time >= p.price_start_time     AND u.usage_end_time <  COALESCE(p.price_end_time, TIMESTAMP'2999-01-01')    WHERE u.usage_metadata.warehouse_id IS NOT NULL      AND u.usage_start_time >= '{START}' AND u.usage_start_time < '{END}'      AND u.billing_origin_product = 'SQL'      {WS_PRED.replace('workspace_id', 'u.workspace_id')}    GROUP BY u.usage_metadata.warehouse_id""")cost.createOrReplaceTempView("wh_cost")display(cost.orderBy(F.desc("dbu_cost_usd")).limit(20))

In [ ]:
# Classic infra cost (EC2/EBS) - NOT in system tables.if CUR_TABLE:    infra = spark.sql(f"""        SELECT warehouse_id, SUM(infra_cost_usd) AS infra_cost_usd        FROM {CUR_TABLE}        WHERE usage_date >= DATE'{START[:10]}' AND usage_date < DATE'{END[:10]}'        GROUP BY warehouse_id    """)    INFRA_SOURCE = "cur"else:    # Fallback: estimate Classic infra as a multiple of DBU cost. Serverless has none.    infra = spark.sql(f"""        SELECT c.warehouse_id,               CASE WHEN w.warehouse_type = 'SERVERLESS' THEN 0.0                    ELSE c.dbu_cost_usd * {INFRA_MULT} END AS infra_cost_usd        FROM wh_cost c JOIN wh_config w USING (warehouse_id)    """)    INFRA_SOURCE = "estimated"    print("WARN: no CUR table supplied. Classic infra cost is ESTIMATED.")    print("      Classic-vs-Serverless comparisons are biased toward Classic until this is fixed.")infra.createOrReplaceTempView("wh_infra")print(f"infra cost source: {INFRA_SOURCE}")

### 1d · Lifecycle → cluster-hours (Tier 2)The billable unit is `∫ cluster_count dt`, **not** uptime. Costing a 4-clusterwarehouse as if it ran one cluster understates spend ~4x.

In [ ]:
if AVAIL["warehouse_events"]:    life = spark.sql(f"""        WITH ev AS (          SELECT warehouse_id, event_time, event_type,                 COALESCE(CAST(cluster_count AS INT), 0) AS cluster_count,                 LEAD(event_time) OVER (PARTITION BY warehouse_id ORDER BY event_time) AS next_time          FROM system.compute.warehouse_events          WHERE event_time >= '{START}' AND event_time < '{END}' {WS_PRED}        ),        spans AS (          SELECT warehouse_id, event_type, cluster_count,                 (UNIX_TIMESTAMP(LEAST(COALESCE(next_time, TIMESTAMP'{END}'), TIMESTAMP'{END}'))                  - UNIX_TIMESTAMP(event_time)) / 3600.0 AS hrs          FROM ev        )        SELECT warehouse_id,               SUM(CASE WHEN event_type <> 'STOPPED' THEN hrs ELSE 0 END)                  AS uptime_hours,               SUM(CASE WHEN event_type <> 'STOPPED' THEN hrs * cluster_count ELSE 0 END)  AS cluster_hours,               MAX(cluster_count)                                                          AS max_cluster_count,               SUM(CASE WHEN event_type = 'STARTING' THEN 1 ELSE 0 END)                    AS cold_starts,               SUM(CASE WHEN event_type IN ('SCALED_UP','SCALED_DOWN') THEN 1 ELSE 0 END)  AS scale_events        FROM spans WHERE hrs >= 0        GROUP BY warehouse_id    """)else:    life = spark.sql("""        SELECT warehouse_id, CAST(NULL AS DOUBLE) uptime_hours, CAST(NULL AS DOUBLE) cluster_hours,               CAST(NULL AS INT) max_cluster_count, CAST(NULL AS INT) cold_starts,               CAST(NULL AS INT) scale_events        FROM wh_cost    """)life.createOrReplaceTempView("wh_life")display(life.orderBy(F.desc("cluster_hours")).limit(20))

In [ ]:
# Duration-weighted p95 of concurrent CLUSTERS (an unweighted percentile# over-provisions off transient spikes).if AVAIL["warehouse_events"]:    ccl = spark.sql(f"""        WITH ev AS (          SELECT warehouse_id, event_time, event_type,                 COALESCE(CAST(cluster_count AS INT),0) cluster_count,                 LEAD(event_time) OVER (PARTITION BY warehouse_id ORDER BY event_time) next_time          FROM system.compute.warehouse_events          WHERE event_time >= '{START}' AND event_time < '{END}' {WS_PRED}        ),        spans AS (          SELECT warehouse_id, cluster_count,                 (UNIX_TIMESTAMP(LEAST(COALESCE(next_time, TIMESTAMP'{END}'), TIMESTAMP'{END}'))                  - UNIX_TIMESTAMP(event_time)) AS secs          FROM ev WHERE event_type <> 'STOPPED'        ),        agg AS (          SELECT warehouse_id, cluster_count, SUM(secs) secs FROM spans          WHERE secs > 0 GROUP BY warehouse_id, cluster_count        ),        cum AS (          SELECT warehouse_id, cluster_count, secs,                 SUM(secs) OVER (PARTITION BY warehouse_id ORDER BY cluster_count                                 ROWS UNBOUNDED PRECEDING) / SUM(secs) OVER (PARTITION BY warehouse_id) frac,                 SUM(secs) OVER (PARTITION BY warehouse_id) total_secs          FROM agg        )        SELECT warehouse_id,               MIN(CASE WHEN frac >= 0.50 THEN cluster_count END) p50_cluster_count,               MIN(CASE WHEN frac >= 0.95 THEN cluster_count END) p95_cluster_count,               SUM(CASE WHEN cluster_count = (SELECT MAX(cluster_count) FROM agg a                                              WHERE a.warehouse_id = cum.warehouse_id)                        THEN secs ELSE 0 END) / MAX(total_secs) AS time_at_max_pct        FROM cum GROUP BY warehouse_id    """)else:    ccl = spark.sql("""SELECT warehouse_id, CAST(NULL AS INT) p50_cluster_count,                       CAST(NULL AS INT) p95_cluster_count, CAST(NULL AS DOUBLE) time_at_max_pct                       FROM wh_cost""")ccl.createOrReplaceTempView("wh_clusters")display(ccl.limit(20))

### 1e · Query signals (Tier 3)`spilled_local_bytes` is the safety metric — it is the only reliable memory-pressuresignal for SQL warehouses, since node CPU/memory is not exposed.

In [ ]:
q = spark.sql(f"""    SELECT compute.warehouse_id AS warehouse_id,           COUNT(*)                                                          AS query_count,           COUNT(DISTINCT executed_by)                                       AS distinct_users,           COUNT(DISTINCT DATE(start_time))                                  AS active_days,           SUM(execution_duration_ms)      / 3600000.0                       AS exec_hours,           SUM(total_duration_ms)          / 3600000.0                       AS total_hours,           AVG(CASE WHEN waiting_at_capacity_duration_ms > 0 THEN 1.0 ELSE 0.0 END)  AS queue_rate,           PERCENTILE(waiting_at_capacity_duration_ms, 0.95)                 AS p95_queue_ms,           AVG(CASE WHEN waiting_for_compute_duration_ms > 0 THEN 1.0 ELSE 0.0 END)  AS coldstart_rate,           PERCENTILE(waiting_for_compute_duration_ms, 0.95)                 AS p95_coldstart_ms,           AVG(CASE WHEN {C_SPILL} > 0 THEN 1.0 ELSE 0.0 END)                AS spill_rate,           SUM({C_SPILL})                                                    AS spill_bytes,           AVG(CASE WHEN from_result_cache THEN 1.0 ELSE 0.0 END)            AS result_cache_rate,           AVG(read_io_cache_percent)                                        AS avg_io_cache_pct,           AVG(CASE WHEN execution_status <> 'FINISHED' THEN 1.0 ELSE 0.0 END) AS failed_rate,           AVG(CASE WHEN statement_type IN ('INSERT','MERGE','UPDATE','DELETE','COPY')                    THEN 1.0 ELSE 0.0 END)                                   AS write_stmt_rate,           SUM(read_bytes)                                                   AS read_bytes,           SUM({C_READ_FILES})                                               AS read_files,           SUM({C_PRUNED})                                                   AS pruned_files,           MAX(start_time)                                                   AS last_query_at    FROM system.query.history    WHERE start_time >= '{START}' AND start_time < '{END}'      AND compute.warehouse_id IS NOT NULL      {WS_PRED}    GROUP BY compute.warehouse_id""")q.createOrReplaceTempView("wh_query")display(q.orderBy(F.desc("query_count")).limit(20))

### 1f · True concurrency (sweep-line)Concurrent **queries** via interval overlap, duration-weighted. Drives `max_clusters`:`required_clusters = ceil(p95_concurrency / 10)`.

In [ ]:
conc = spark.sql(f"""    WITH edges AS (      SELECT compute.warehouse_id wh, start_time ts,  1 delta FROM system.query.history      WHERE start_time >= '{START}' AND start_time < '{END}' AND compute.warehouse_id IS NOT NULL {WS_PRED}      UNION ALL      SELECT compute.warehouse_id, end_time, -1 FROM system.query.history      WHERE start_time >= '{START}' AND start_time < '{END}' AND compute.warehouse_id IS NOT NULL {WS_PRED}    ),    running AS (      SELECT wh, ts,             SUM(delta) OVER (PARTITION BY wh ORDER BY ts, delta DESC ROWS UNBOUNDED PRECEDING) c      FROM edges    ),    spans AS (      SELECT wh, c,             UNIX_TIMESTAMP(LEAD(ts) OVER (PARTITION BY wh ORDER BY ts)) - UNIX_TIMESTAMP(ts) secs      FROM running    ),    agg AS (      SELECT wh, c, SUM(secs) secs FROM spans WHERE secs > 0 AND c >= 0 GROUP BY wh, c    ),    cum AS (      SELECT wh, c, SUM(secs) OVER (PARTITION BY wh ORDER BY c ROWS UNBOUNDED PRECEDING)                    / SUM(secs) OVER (PARTITION BY wh) frac      FROM agg    )    SELECT wh AS warehouse_id,           MIN(CASE WHEN frac >= 0.95 THEN c END) AS p95_concurrency,           MAX(c)                                 AS peak_concurrency    FROM cum GROUP BY wh""")conc.createOrReplaceTempView("wh_conc")display(conc.orderBy(F.desc("peak_concurrency")).limit(20))

## 2 · Derive the decision signals**Duty cycle** = query execution time / available cluster-time. It is the primaryinput to the Classic-vs-Serverless decision.

In [ ]:
signals = spark.sql(f"""    SELECT w.*,           COALESCE(c.dbu_cost_usd, 0)                       AS dbu_cost_usd,           COALESCE(i.infra_cost_usd, 0)                     AS infra_cost_usd,           COALESCE(c.dbu_cost_usd, 0) + COALESCE(i.infra_cost_usd, 0) AS total_cost_usd,           c.dbus, l.uptime_hours, l.cluster_hours, l.cold_starts, l.scale_events,           cl.p50_cluster_count, cl.p95_cluster_count, cl.time_at_max_pct,           COALESCE(q.query_count, 0)  AS query_count,           q.distinct_users, q.active_days, q.exec_hours, q.total_hours,           COALESCE(q.queue_rate, 0)   AS queue_rate,           q.p95_queue_ms,           COALESCE(q.coldstart_rate,0) AS coldstart_rate, q.p95_coldstart_ms,           q.spill_rate, q.spill_bytes,           COALESCE(q.result_cache_rate,0) AS result_cache_rate,           q.avg_io_cache_pct,           COALESCE(q.failed_rate,0)   AS failed_rate,           COALESCE(q.write_stmt_rate,0) AS write_stmt_rate,           q.read_bytes, q.read_files, q.pruned_files, q.last_query_at,           cc.p95_concurrency, cc.peak_concurrency,           -- derived --------------------------------------------------------           CASE WHEN l.cluster_hours > 0 THEN q.exec_hours / l.cluster_hours END      AS duty_cycle,           CASE WHEN l.uptime_hours > 0                THEN GREATEST(0, l.uptime_hours - COALESCE(q.total_hours,0)) / l.uptime_hours END AS idle_ratio,           CASE WHEN l.cluster_hours > 0                THEN COALESCE(c.dbu_cost_usd,0) *                     (GREATEST(0, l.uptime_hours - COALESCE(q.total_hours,0)) / l.uptime_hours) END AS idle_cost_usd,           CASE WHEN q.query_count > 0                THEN (COALESCE(c.dbu_cost_usd,0)+COALESCE(i.infra_cost_usd,0)) / q.query_count END AS cost_per_query,           COALESCE(c.dbu_cost_usd,0) * COALESCE(q.failed_rate,0)                     AS failed_query_waste_usd,           CASE WHEN q.read_files > 0 THEN q.pruned_files / q.read_files END          AS prune_ratio,           CEIL(COALESCE(cc.p95_concurrency,1) / {QUERIES_PER_CLUSTER}.0)             AS required_clusters,           CASE WHEN l.uptime_hours > 0 THEN l.cold_starts / (l.uptime_hours/24) END  AS cold_starts_per_day    FROM wh_config w    LEFT JOIN wh_cost     c  USING (warehouse_id)    LEFT JOIN wh_infra    i  USING (warehouse_id)    LEFT JOIN wh_life     l  USING (warehouse_id)    LEFT JOIN wh_clusters cl USING (warehouse_id)    LEFT JOIN wh_query    q  USING (warehouse_id)    LEFT JOIN wh_conc     cc USING (warehouse_id)""")signals.createOrReplaceTempView("wh_signals")sdf = signals.toPandas()print(f"{len(sdf)} warehouses  |  total spend ${sdf.total_cost_usd.sum():,.0f}")display(signals.orderBy(F.desc("total_cost_usd")))

## 3 · Decision engineThe config space is ~9,450 points — small enough to **enumerate exhaustively**and pick the cheapest feasible option. No heuristic search, no model.### Cost model```cost = size_dbu x clusters x hours x $/DBU   (+ infra, Classic only)```Split into two components that behave differently under a size change:| Component | Under size change ||---|---|| **busy** cluster-hours | ~cost-neutral (2x size ≈ 1/2 runtime) || **idle** cluster-hours | scales **linearly** with size |This is the honest model, and it has a blunt consequence: **downsizing only savesmoney on idle-dominated warehouses.** On a throughput-bound warehouse, downsizingsaves nothing and just makes queries slower. Cutting idle and cluster count iswhere the money actually is.

In [ ]:
def size_idx(s):    return SIZE_ORDER.index(s) if s in SIZE_ORDER else Nonedef dbu_rate_for(wtype):    """$/DBU by warehouse type, derived from observed billing where possible."""    return RATE_BY_TYPE.get(wtype, RATE_BY_TYPE.get("CLASSIC", 0.22))# Observed effective $/DBU per warehouse type (list price, from this window)._r = spark.sql("""    SELECT CASE WHEN sku_name ILIKE '%SERVERLESS%' THEN 'SERVERLESS'                WHEN sku_name ILIKE '%PRO%'        THEN 'PRO' ELSE 'CLASSIC' END wtype,           SUM(dbu_cost_usd)/NULLIF(SUM(dbus),0) rate    FROM wh_cost GROUP BY 1""").toPandas()RATE_BY_TYPE = {r.wtype: float(r.rate) for r in _r.itertuples() if r.rate and r.rate > 0}RATE_BY_TYPE.setdefault("CLASSIC", 0.22)RATE_BY_TYPE.setdefault("PRO", 0.55)RATE_BY_TYPE.setdefault("SERVERLESS", 0.70)print("observed $/DBU:", {k: round(v, 4) for k, v in RATE_BY_TYPE.items()})

In [ ]:
def score_config(row, wtype, size, min_cl, max_cl, auto_stop):    """Cost of one candidate config over the window. Returns (cost, feasible, reasons)."""    reasons = []    cur_size = row.warehouse_size if row.warehouse_size in SIZE_DBU else "Medium"    cur_dbu, new_dbu = SIZE_DBU[cur_size], SIZE_DBU[size]    ratio = new_dbu / cur_dbu    busy_h = row.exec_hours if pd.notna(row.exec_hours) else 0.0    up_h   = row.uptime_hours if pd.notna(row.uptime_hours) else 0.0    wall_h = row.total_hours if pd.notna(row.total_hours) else busy_h    idle_h = max(0.0, up_h - wall_h)    # --- feasibility -------------------------------------------------------    req = int(row.required_clusters) if pd.notna(row.required_clusters) else 1    if max_cl < req:        reasons.append(f"max_clusters {max_cl} < required {req}")    # Never downsize into (or deeper into) memory pressure.    if ratio < 1.0:        if not HAS_SPILL or pd.isna(row.spill_rate):            reasons.append("no spill data - downsize unsafe")        elif row.spill_rate > TH["spill_rate_safe_downsize"]:            reasons.append(f"spill_rate {row.spill_rate:.1%} too high to downsize")    # Existing memory pressure demands a size increase.    if HAS_SPILL and pd.notna(row.spill_rate) and row.spill_rate > TH["spill_rate_upsize"] and ratio <= 1.0:        reasons.append("spill requires larger size")    # --- cost --------------------------------------------------------------    # busy: work is ~conserved, so cluster-hours shrink as size grows    busy_cluster_h = busy_h / max(ratio, 1e-9)    # idle: bounded by auto_stop, paid at min_clusters (warm floor) and scales with size    stop_frac  = min(1.0, auto_stop / max(row.auto_stop_minutes or 10, 1))    idle_cl_h  = idle_h * stop_frac * max(min_cl, 1)    # warm floor: min_clusters billed for the whole uptime    floor_h    = up_h * max(min_cl - 1, 0)    # cold starts: Classic bills startup; Serverless is ~free    starts     = row.cold_starts_per_day * (LOOKBACK_DAYS) if pd.notna(row.cold_starts_per_day) else 0    start_min  = TH["cold_start_min_serverless"] if wtype == "SERVERLESS" else TH["cold_start_min_classic"]    # tighter auto_stop => more restarts    start_mult = (row.auto_stop_minutes or 10) / max(auto_stop, 1)    start_h    = starts * start_mult * start_min / 60.0    cluster_h  = busy_cluster_h + idle_cl_h + floor_h + start_h    dbu_cost   = new_dbu * cluster_h * dbu_rate_for(wtype)    # Classic carries EC2/EBS on top; Serverless does not.    if wtype == "SERVERLESS":        infra = 0.0    elif CUR_TABLE and row.infra_cost_usd and row.dbu_cost_usd:        infra = dbu_cost * (row.infra_cost_usd / max(row.dbu_cost_usd, 1e-9))    else:        infra = dbu_cost * INFRA_MULT    return dbu_cost + infra, (len(reasons) == 0), reasonsdef enumerate_best(row):    """Exhaustive scan of the config space. ~9.4k points, milliseconds."""    cur_size = row.warehouse_size if row.warehouse_size in SIZE_DBU else "Medium"    cur_i    = size_idx(cur_size)    req      = int(row.required_clusters) if pd.notna(row.required_clusters) else 1    types      = ["SERVERLESS", "CLASSIC", "PRO"]    sizes      = SIZE_ORDER[max(0, cur_i - 2): min(len(SIZE_ORDER), cur_i + 3)]  # +/-2 steps    min_cls    = [1, 2, 3]    max_cls    = sorted({req, req + 1, max(req, 2), 4, 6, 8, 10,                         int(row.max_clusters) if pd.notna(row.max_clusters) else 4})    stops      = [1, 5, 10, 15, 30, 45, 60]    best, evaluated = None, 0    for t in types:        for s in sizes:            for mn in min_cls:                for mx in max_cls:                    if mn > mx:                        continue                    for st in stops:                        if t == "SERVERLESS" and mn > 1:                            continue  # warm floor is pure waste when startup is seconds                        c, ok, why = score_config(row, t, s, mn, mx, st)                        evaluated += 1                        if ok and (best is None or c < best[0]):                            best = (c, t, s, mn, mx, st)    return best, evaluatedprint("engine ready")

In [ ]:
recs = []for row in sdf.itertuples():    cur_cost = float(row.total_cost_usd or 0)    flags, conf = [], "high"    # ---- zombie: no queries but still billing -----------------------------    if (row.query_count or 0) == 0 and cur_cost > 0:        recs.append(dict(warehouse_id=row.warehouse_id, warehouse_name=row.warehouse_name,                         action="DELETE_OR_STOP", current_cost_usd=cur_cost,                         recommended_cost_usd=0.0, savings_usd=cur_cost, savings_pct=100.0,                         rec_type=None, rec_size=None, rec_min_clusters=None,                         rec_max_clusters=None, rec_auto_stop=None,                         flags="zombie_warehouse", confidence="high",                         rationale=f"No queries in {LOOKBACK_DAYS}d but ${cur_cost:,.0f} billed. "                                   f"Warehouse appears abandoned."))        continue    best, n = enumerate_best(row)    if best is None:        recs.append(dict(warehouse_id=row.warehouse_id, warehouse_name=row.warehouse_name,                         action="NO_CHANGE", current_cost_usd=cur_cost,                         recommended_cost_usd=cur_cost, savings_usd=0.0, savings_pct=0.0,                         rec_type=None, rec_size=None, rec_min_clusters=None,                         rec_max_clusters=None, rec_auto_stop=None,                         flags="no_feasible_config", confidence="low",                         rationale="No candidate config satisfied the constraints."))        continue    new_cost, t, s, mn, mx, st = best    savings = cur_cost - new_cost    # ---- guardrails: suppress recommendations a missing signal could invert    if not HAS_SPILL:        flags.append("no_spill_data")        conf = "low"    if row.warehouse_type != "SERVERLESS" and t != "SERVERLESS" and INFRA_SOURCE == "estimated":        flags.append("classic_cost_estimated")   # stay-Classic without CUR is unsafe        conf = "low"    if (row.active_days or 0) < TH["min_days_for_high_conf"] or \       (row.query_count or 0) < TH["min_queries_for_high_conf"]:        flags.append("insufficient_history")        conf = "low" if conf == "low" else "medium"    if pd.notna(row.spill_rate) and row.spill_rate > TH["spill_rate_upsize"]:        flags.append("memory_pressure")    if (row.queue_rate or 0) > TH["queue_rate_scale_out"]:        flags.append("queuing")    if (row.result_cache_rate or 0) > 0.30 and st < (row.auto_stop_minutes or 10):        flags.append("cache_loss_risk")    if (row.write_stmt_rate or 0) > 0.50:        flags.append("etl_on_warehouse")    if (row.failed_rate or 0) > 0.15:        flags.append("high_failure_rate")    if pd.notna(row.min_clusters) and row.min_clusters > 1 and (row.queue_rate or 0) < 0.02:        flags.append("unjustified_warm_floor")    changed = []    if t  != row.warehouse_type: changed.append(f"{row.warehouse_type}->{t}")    if s  != row.warehouse_size: changed.append(f"size {row.warehouse_size}->{s}")    if pd.notna(row.min_clusters) and mn != row.min_clusters: changed.append(f"min_cl {row.min_clusters}->{mn}")    if pd.notna(row.max_clusters) and mx != row.max_clusters: changed.append(f"max_cl {row.max_clusters}->{mx}")    if pd.notna(row.auto_stop_minutes) and st != row.auto_stop_minutes: changed.append(f"auto_stop {row.auto_stop_minutes}->{st}")    recs.append(dict(        warehouse_id=row.warehouse_id, warehouse_name=row.warehouse_name,        action="RECONFIGURE" if changed else "NO_CHANGE",        current_cost_usd=cur_cost, recommended_cost_usd=new_cost,        savings_usd=max(savings, 0.0),        savings_pct=(100.0 * savings / cur_cost) if cur_cost > 0 else 0.0,        rec_type=t, rec_size=s, rec_min_clusters=mn, rec_max_clusters=mx, rec_auto_stop=st,        flags=",".join(flags), confidence=conf,        rationale="; ".join(changed) if changed else "Current configuration is already optimal."))rdf = pd.DataFrame(recs)print(f"{len(rdf)} recommendations | potential savings ${rdf.savings_usd.sum():,.0f}/{LOOKBACK_DAYS}d")

## 4 · Rationale (template — deterministic)Deliberately templated, not LLM-generated. The rationale must reproduce exactlyfor the same inputs so a platform engineer can audit it. Swap in an LLM later**for prose quality only** — it must never alter a number or a decision.

In [ ]:
def rationale_for(sig, rec):    p = []    if pd.notna(sig.duty_cycle):        p.append(f"Duty cycle {sig.duty_cycle:.0%} over {LOOKBACK_DAYS}d "                 f"({sig.query_count:,} queries, {sig.exec_hours:,.0f} exec-hrs on "                 f"{sig.cluster_hours:,.0f} cluster-hrs).")    if pd.notna(sig.idle_ratio) and sig.idle_ratio > TH["idle_ratio_autostop"]:        p.append(f"Idle {sig.idle_ratio:.0%} of uptime (~${sig.idle_cost_usd or 0:,.0f}).")    if rec.rec_type and rec.rec_type != sig.warehouse_type:        why = ("low utilisation makes per-second billing and instant start cheaper"               if rec.rec_type == "SERVERLESS"               else "sustained load amortises the lower DBU rate")        p.append(f"Duty cycle favours {rec.rec_type}: {why}.")    if pd.notna(sig.p95_concurrency):        p.append(f"p95 concurrency {sig.p95_concurrency:.0f} queries -> "                 f"{sig.required_clusters:.0f} cluster(s) needed vs max_clusters={sig.max_clusters}.")    if "queuing" in (rec.flags or ""):        p.append(f"Queuing on {sig.queue_rate:.0%} of queries (p95 {(sig.p95_queue_ms or 0)/1000:.1f}s) "                 f"-> scale OUT, not up.")    if "memory_pressure" in (rec.flags or ""):        p.append(f"Spill on {sig.spill_rate:.0%} of queries -> scale UP required.")    if "unjustified_warm_floor" in (rec.flags or ""):        p.append(f"min_clusters={sig.min_clusters} bills 24/7 with no queuing to justify it.")    if "etl_on_warehouse" in (rec.flags or ""):        p.append(f"{sig.write_stmt_rate:.0%} write statements - consider moving this ETL to a Job cluster.")    if "cache_loss_risk" in (rec.flags or ""):        p.append(f"{sig.result_cache_rate:.0%} result-cache hits - tightening auto_stop may regress latency.")    if "classic_cost_estimated" in (rec.flags or ""):        p.append("WARNING: Classic infra cost is ESTIMATED (no CUR). Verify before acting.")    if "no_spill_data" in (rec.flags or ""):        p.append("WARNING: no spill data - downsize suppressed.")    return " ".join(p)# Key on warehouse_id rather than assigning by position - merge order is not# guaranteed to match rdf's row order.merged = rdf.merge(sdf, on="warehouse_id", suffixes=("", "_sig"))_rat = {r.warehouse_id: rationale_for(r, r) for r in merged.itertuples()}rdf["rationale"] = rdf.warehouse_id.map(_rat).fillna(rdf["rationale"])rdf = rdf.sort_values("savings_usd", ascending=False).reset_index(drop=True)

## 5 · Output

In [ ]:
total_cur  = rdf.current_cost_usd.sum()total_save = rdf.savings_usd.sum()act        = rdf[rdf.action != "NO_CHANGE"]print(f"""================= SQL WAREHOUSE OPTIMIZATION ================= window            {LOOKBACK_DAYS} days ({START[:10]} -> {END[:10]}) warehouses        {len(rdf)} current spend     ${total_cur:,.0f} potential saving  ${total_save:,.0f}  ({100*total_save/max(total_cur,1):.1f}%) actionable        {len(act)} infra cost source {INFRA_SOURCE.upper()}{'  <-- ESTIMATED, verify before acting' if INFRA_SOURCE=='estimated' else ''} spill data        {'available' if HAS_SPILL else 'MISSING - downsizes suppressed'}==============================================================""")display(spark.createDataFrame(rdf[[    "warehouse_name","action","current_cost_usd","recommended_cost_usd","savings_usd","savings_pct",    "rec_type","rec_size","rec_min_clusters","rec_max_clusters","rec_auto_stop",    "confidence","flags","rationale"]]))

In [ ]:
if OUTPUT_TABLE:    out = spark.createDataFrame(rdf) \        .withColumn("generated_at", F.current_timestamp()) \        .withColumn("lookback_days", F.lit(LOOKBACK_DAYS)) \        .withColumn("infra_source", F.lit(INFRA_SOURCE))    out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)    print(f"written -> {OUTPUT_TABLE}")else:    print("output_table blank - display only")

## Caveats — read before acting on any number1. **Classic infra cost.** Without a real CUR join, `classic_infra_multiplier` is a   guess. DBUs are only ~40–60% of true Classic cost, so an under-estimate biases   every recommendation toward staying Classic. Any such recommendation is flagged   `classic_cost_estimated` and forced to `confidence=low`.2. **`list_prices` is list price.** Override `RATE_BY_TYPE` with your negotiated   rate or savings are over-stated.3. **The cost model is first-order.** It assumes work is conserved under a size   change (2x size ≈ 1/2 runtime) and that idle scales linearly with size. Good   enough to rank and to size the prize; not a substitute for a replay simulator   if you need per-warehouse precision.4. **Schema drift.** `query.history` column names have changed across releases —   the preflight cell resolves them, but check its output.5. **`system.compute.warehouses` may not exist** in your region yet. The fallback   infers only `warehouse_type` from SKU; every config-based rule silently   degrades. Check the preflight warning.6. **Savings do not sum cleanly.** Type, size and cluster changes compound   multiplicatively. This notebook prices the *whole config as one unit*, which is   correct — but do not decompose the total into per-lever attribution without   re-scoring each lever independently.### Next stepReplace `score_config` with a discrete-event replay simulator: replay the actualquery arrival timeline against each candidate config (10 queries/cluster admission,startup latency, auto-stop). Deterministic, testable, and materially more accuratethan this closed-form approximation.